<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/NB05_evolutionary_design.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 05: Evolutionary Computation for Engineering Design

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook defines the beam design problem from the formulas of mechanics of materials [1], derives its optimum by hand, and then solves it with a real-coded genetic algorithm written from scratch and with SciPy's differential evolution [2]. It compares penalty handling with Deb's feasibility rules, studies convergence over ten seeds and builds a Pareto front of mass against deflection by repeated constrained optimisation.

This notebook is the hands-on part of Week 5. It opens with Python step 5: Random numbers, arrays and genetic operators and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-05/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/Week05_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-05/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 5 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, NonlinearConstraint

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 5: Random numbers, arrays and genetic operators

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Reproducible randomness

Evolutionary algorithms draw random numbers at every step, so repeated runs give different results. A random number generator created with `np.random.default_rng(seed)` produces the same sequence every time it starts from the same seed. Fixed seeds make experiments repeatable, and comparing algorithms over several seeds shows how much of a difference is due to chance.

In [ ]:
import numpy as np

gen = np.random.default_rng(42)
print(gen.uniform(0.1, 0.5, size=3))      # three numbers between 0.1 and 0.5
print(gen.integers(1, 7, size=5))         # five dice rolls from 1 to 6
again = np.random.default_rng(42)
print(again.uniform(0.1, 0.5, size=3))    # the same three numbers as the first line

### A population as a two-dimensional array

A population of candidate designs fits in a two-dimensional array: Each row is one design, and each column is one design variable. For a rectangular steel beam, the columns hold the width b and the height h of the section. `pop[:, 0]` selects the first column of all rows, and formulas written with arrays evaluate the whole population at once. The beam below spans 5 m and carries a bending moment of 25 kN m, and the bending stress of a rectangular section is 6 M / (b h^2).

In [ ]:
L_m, M_Nm, rho, allow_Pa = 5.0, 25e3, 7850.0, 150e6
pop = gen.uniform([0.02, 0.05], [0.20, 0.40], size=(8, 2))   # columns: b and h in metres
b, h = pop[:, 0], pop[:, 1]
mass = rho * b * h * L_m                   # kg
stress = 6 * M_Nm / (b * h ** 2)           # Pa
feasible = stress <= allow_Pa
print(pop.shape, feasible)
print(np.round(mass, 1))

`25e3` is scientific notation for 25000. The bounds are given per column, so the generator draws widths between 0.02 and 0.20 m and heights between 0.05 and 0.40 m.

### Sorting and selection

`np.argsort` returns the positions that would sort an array from the smallest to the largest value. Indexing an array with these positions reorders its rows, and slicing keeps the best few. Infeasible designs receive a penalty that grows with the violation of the stress limit, a common way to handle constraints in evolutionary search.

In [ ]:
violation = np.maximum(stress / allow_Pa - 1, 0)    # 0 for feasible designs
fitness = mass * (1 + 10 * violation)               # penalised mass
order = np.argsort(fitness)                         # positions from best to worst
best3 = pop[order[:3]]
print(order)
print(np.round(best3, 3))

### Comprehensions and lambda functions

A list comprehension builds a list in one line, in the form `[expression for item in sequence if condition]`, and replaces a loop that appends to a list. A lambda is a short function without a name, often used as a sorting key. `min` with the `key` argument returns the item whose key is smallest.

In [ ]:
labels = [f"b={bb:.2f} h={hh:.2f}" for bb, hh in pop[:3]]
print(labels)
light = [i for i in range(len(pop)) if mass[i] < 500]
print("designs lighter than 500 kg:", light)
area = lambda bb, hh: bb * hh
print(np.round(min(pop, key=lambda d: area(d[0], d[1])), 3))

### Crossover, mutation and one generation

Selection alone only reorders existing designs. Genetic algorithms create new ones with two operators [3]. Crossover mixes two parents: For real-valued variables, arithmetic crossover takes a weighted mean, child = alpha parent1 + (1 - alpha) parent2, with a random weight alpha between 0 and 1. Mutation adds a small random change to each variable, and `np.clip` pulls values that leave the allowed range back to the nearest bound.

In [ ]:
low, high = np.array([0.02, 0.05]), np.array([0.20, 0.40])
p1, p2 = pop[order[0]], pop[order[1]]                  # the two best designs as parents
alpha = gen.uniform(0, 1)
child = alpha * p1 + (1 - alpha) * p2                  # arithmetic crossover
child = child + gen.normal(0, [0.005, 0.01])           # Gaussian mutation, one scale per variable
child = np.clip(child, low, high)                      # stay inside the bounds
print(np.round(p1, 3), np.round(p2, 3), "->", np.round(child, 3))

A function that returns the penalised mass of a whole population lets the operators run in a loop. The loop below evolves 20 designs for 60 generations. The best five designs survive unchanged, which is called elitism and guarantees that the best design found so far is never lost, and fifteen children of randomly chosen parents from the better half fill the rest of the population. `np.vstack` stacks the survivors and the children into the next population.

In [ ]:
def penalised_mass(P):
    """Penalised mass in kg of every design (row) of P."""
    m = rho * P[:, 0] * P[:, 1] * L_m
    s = 6 * M_Nm / (P[:, 0] * P[:, 1] ** 2)
    return m * (1 + 10 * np.maximum(s / allow_Pa - 1, 0))


evo = np.random.default_rng(1)
P = evo.uniform(low, high, size=(20, 2))
for g in range(60):
    P = P[np.argsort(penalised_mass(P))]                # best design first
    kids = []
    for _ in range(15):
        a, b_ = P[evo.choice(10, size=2, replace=False)]
        w = evo.uniform(0, 1)
        kids.append(np.clip(w * a + (1 - w) * b_ + evo.normal(0, [0.002, 0.005]), low, high))
    P = np.vstack([P[:5], kids])                        # elitism: the best five survive unchanged
    if g % 20 == 0:
        print(f"generation {g:2d}: best penalised mass {penalised_mass(P).min():.1f} kg")
best = P[np.argmin(penalised_mass(P))]
print(f"best design: b = {best[0]:.3f} m, h = {best[1]:.3f} m, mass {penalised_mass(best[None, :])[0]:.1f} kg")

A hand calculation gives the lightest feasible section of this problem: The width sits at its lower bound of 0.02 m, and the height just satisfies the stress limit, h = 0.224 m, for a mass of 175.5 kg. The genetic algorithm approaches this design from random starting points without any derivative, which is what makes it useful when no formula for the optimum exists.

**Quick check.** What does [x ** 2 for x in range(4) if x % 2 == 0] produce?

A. [0, 4]  
B. [0, 1, 4, 9]  
C. [4]  
D. [1, 9]

<details><summary>Show the answer</summary>

**A.** range(4) yields 0, 1, 2 and 3. Only 0 and 2 are even, and their squares are 0 and 4.

</details>

### Exercises for Python step 5

Generate 1000 designs with their own generator, so that the result does not depend on how often the cells above have been run: `designs = np.random.default_rng(7).uniform([0.02, 0.05], [0.20, 0.40], size=(1000, 2))`.

**Exercise 5.1.** Count the designs whose bending stress stays within 150 MPa and store the count in `py_n_feasible`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py5_designs():
    import numpy as np
    d = np.random.default_rng(7).uniform([0.02, 0.05], [0.20, 0.40], size=(1000, 2))
    s = 6 * 25e3 / (d[:, 0] * d[:, 1] ** 2)
    m = 7850.0 * d[:, 0] * d[:, 1] * 5.0
    return d, s, m


def _py5_n_feasible_reference():
    d, s, m = _py5_designs()
    return int((s <= 150e6).sum())

In [ ]:
designs = np.random.default_rng(7).uniform([0.02, 0.05], [0.20, 0.40], size=(1000, 2))
py_n_feasible = None
check("Exercise 5.1", py_n_feasible, _py5_n_feasible_reference())

**Exercise 5.2.** Among the feasible designs, find the smallest mass in kilograms and store it in `py_best_mass`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py5_best_mass_reference():
    d, s, m = _py5_designs()
    return float(m[s <= 150e6].min())

In [ ]:
py_best_mass = None
check("Exercise 5.2", py_best_mass, _py5_best_mass_reference())

**Exercise 5.3.** Count the designs whose height is at least three times their width and store the count in `py_slender`. Use a boolean mask rather than a loop.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py5_slender_reference():
    d, s, m = _py5_designs()
    return int((d[:, 1] >= 3 * d[:, 0]).sum())

In [ ]:
py_slender = None
check("Exercise 5.3", py_slender, _py5_slender_reference())

**Exercise 5.4.** Two parents have heights of 0.30 m and 0.20 m. Compute the height of their child with arithmetic crossover and alpha = 0.25, where alpha weights the first parent, and store it in `py_child_h`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py5_child_h_reference():
    return 0.25 * 0.30 + 0.75 * 0.20

In [ ]:
py_child_h = None
check("Exercise 5.4", py_child_h, _py5_child_h_reference())

**Exercise 5.5.** Differential evolution builds a mutant from three designs a, b and c as a + F (b - c). With a = [0.10, 0.30], b = [0.06, 0.25], c = [0.08, 0.20] and F = 0.8, compute the mutant with NumPy arrays and store its width, the first element, in `py_de_width`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py5_de_width_reference():
    return 0.10 + 0.8 * (0.06 - 0.08)

In [ ]:
py_de_width = None
check("Exercise 5.5", py_de_width, _py5_de_width_reference())

## 1. The design problem

A steel cantilever of length L = 2 m carries a tip load P = 10 kN. The rectangular cross-section has width b and height h (m). Minimise the mass subject to:

* bending stress 6 P L / (b h^2) at most 160 MPa,
* tip deflection 4 P L^3 / (E b h^3) at most L / 250 = 8 mm, with E = 200 GPa [1],
* a slenderness limit h at most 4 b, a simple guard against lateral instability,
* bounds 0.02 to 0.20 m for b and 0.05 to 0.50 m for h.

**Python move: several return values and dictionaries of constants.** `beam(x)` returns mass and the three constraint values g, where g at most 0 means satisfied.

In [ ]:
P, L, E, RHO = 10e3, 2.0, 200e9, 7850.0
SIGMA_ALLOW, DEFL_ALLOW = 160e6, L / 250
LO, HI = np.array([0.02, 0.05]), np.array([0.20, 0.50])


def beam(x):
    b, h = x
    mass = RHO * b * h * L
    g = np.array([6 * P * L / (b * h**2) / SIGMA_ALLOW - 1,       # stress
                  4 * P * L**3 / (E * b * h**3) / DEFL_ALLOW - 1,  # deflection
                  h / (4 * b) - 1])                                # slenderness
    return mass, g


def violation(x):
    return float(np.sum(np.maximum(0, beam(x)[1])))


print(beam([0.05, 0.20]))

### The optimum by hand

Mass grows with b h, stress and deflection fall faster with h than with b, so the optimum wants h as large as the slenderness limit allows: h = 4 b. With this, the deflection limit gives b^4 = 4 P L^3 / (64 E delta_allow), and the stress constraint turns out to be inactive.

### Your turn, exercise 1

Compute b, h and the minimum mass from these two active constraints and store the mass in `m_star`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _m_star_reference():
    b = (4 * P * L**3 / (64 * E * DEFL_ALLOW)) ** 0.25
    return RHO * b * (4 * b) * L

In [ ]:
m_star = None   # your computation
check("Exercise 1", m_star, _m_star_reference(), tol=1e-4)

## 2. The design space

A contour plot of mass with the infeasible region shaded shows why the optimum lies on the boundary.

In [ ]:
bb, hh = np.meshgrid(np.linspace(0.02, 0.2, 200), np.linspace(0.05, 0.5, 200))
mass_grid = RHO * bb * hh * L
feasible = (6 * P * L / (bb * hh**2) <= SIGMA_ALLOW) & (4 * P * L**3 / (E * bb * hh**3) <= DEFL_ALLOW) & (hh <= 4 * bb)
plt.figure(figsize=(5.5, 4.2))
cs = plt.contour(bb, hh, mass_grid, levels=[50, 100, 150, 250, 400, 600], colors="gray", linewidths=0.8)
plt.clabel(cs, fmt="%d kg", fontsize=7)
plt.contourf(bb, hh, ~feasible, levels=[0.5, 1.5], colors=["#B53F2E"], alpha=0.18)
plt.xlabel("width b (m)"); plt.ylabel("height h (m)"); plt.title("Mass contours, infeasible region shaded"); plt.show()

## 3. A real-coded genetic algorithm

**Python move: NumPy random generator and argsort.** The algorithm uses tournament selection, blend crossover, Gaussian mutation scaled to the variable range, bounds by clipping and elitism [4, 5]. The fitness is the mass plus a penalty for violation.

In [ ]:
def penalised(x, weight=1e4):
    return beam(x)[0] + weight * violation(x)


def genetic_algorithm(f, lo, hi, pop_size=40, generations=120, mutation=0.08, tournament=2, elite=2, seed=0):
    rng = np.random.default_rng(seed)
    span = hi - lo
    pop = lo + span * rng.random((pop_size, len(lo)))
    history = []
    for _ in range(generations):
        cost = np.array([f(x) for x in pop])
        order = np.argsort(cost)
        history.append(cost[order[0]])
        children = [pop[i].copy() for i in order[:elite]]                      # elitism
        while len(children) < pop_size:
            parents = []
            for _ in range(2):                                                  # tournament selection
                contenders = rng.integers(0, pop_size, tournament)
                parents.append(pop[contenders[np.argmin(cost[contenders])]])
            w = rng.uniform(-0.25, 1.25, len(lo))                              # blend crossover
            child = w * parents[0] + (1 - w) * parents[1]
            child += mutation * span * rng.normal(size=len(lo))                 # Gaussian mutation
            children.append(np.clip(child, lo, hi))
        pop = np.array(children)
    cost = np.array([f(x) for x in pop])
    return pop[np.argmin(cost)], np.array(history)


best, hist = genetic_algorithm(penalised, LO, HI, seed=1)
print("best design b, h =", np.round(best, 4), "mass", round(beam(best)[0], 2), "violation", round(violation(best), 5))

## 4. One run proves little: ten seeds

**Python move: loops over seeds and summary statistics.** A stochastic algorithm is judged by its distribution of results.

In [ ]:
finals = []
plt.figure(figsize=(6.5, 3.4))
for seed in range(10):
    x, h = genetic_algorithm(penalised, LO, HI, seed=seed)
    finals.append(beam(x)[0] if violation(x) < 1e-6 else np.nan)
    plt.plot(h, lw=1, alpha=0.8)
plt.axhline(_m_star_reference(), color="k", ls="--", lw=0.8, label="analytic optimum")
plt.ylim(100, 200); plt.xlabel("generation"); plt.ylabel("best penalised mass (kg)"); plt.legend(); plt.show()
finals = np.array(finals)
print(f"feasible runs {np.sum(~np.isnan(finals))}/10, best {np.nanmin(finals):.2f}, median {np.nanmedian(finals):.2f}, worst {np.nanmax(finals):.2f} kg")

## 5. Differential evolution in SciPy

SciPy's `differential_evolution` accepts bounds and nonlinear constraints directly [2, 6]. The constraint object says that every component of g must lie at or below zero.

In [ ]:
constraint = NonlinearConstraint(lambda x: beam(x)[1], -np.inf, 0.0)
res = differential_evolution(lambda x: beam(x)[0], bounds=list(zip(LO, HI)), constraints=(constraint,), seed=3, tol=1e-10, polish=False)
print("DE design", np.round(res.x, 5), "mass", round(res.fun, 3), "evaluations", res.nfev)

### Your turn, exercise 2

Implement Deb's feasibility rules as a comparison function `better(x, y)` that returns True when design x beats design y [7]: A feasible design beats an infeasible one, two feasible designs are compared by mass, and two infeasible designs by violation. Use it to find the best of the 200 random designs below and store its mass in `best_random_mass`.

In [ ]:
rng = np.random.default_rng(7)
candidates = LO + (HI - LO) * rng.random((200, 2))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _best_random_mass_reference():
    feasible = [x for x in candidates if violation(x) == 0]
    if feasible:
        return min(beam(x)[0] for x in feasible)
    return beam(min(candidates, key=violation))[0]

In [ ]:
best_random_mass = None   # your computation
check("Exercise 2", best_random_mass, _best_random_mass_reference())

## 6. Two objectives: mass and deflection

Lighter beams deflect more. Minimising mass for a sequence of deflection limits traces the Pareto front [8].

In [ ]:
front = []
for limit_mm in [2, 3, 4, 5, 6, 8, 10, 14, 20]:
    lim = limit_mm / 1000
    cons = NonlinearConstraint(lambda x, lim=lim: np.array([6 * P * L / (x[0] * x[1]**2) / SIGMA_ALLOW - 1,
                                                          4 * P * L**3 / (E * x[0] * x[1]**3) / lim - 1,
                                                          x[1] / (4 * x[0]) - 1]), -np.inf, 0.0)
    r = differential_evolution(lambda x: RHO * x[0] * x[1] * L, bounds=list(zip(LO, HI)), constraints=(cons,), seed=0, tol=1e-9, polish=False)
    front.append((r.fun, 1000 * 4 * P * L**3 / (E * r.x[0] * r.x[1]**3)))
front = np.array(front)
plt.figure(figsize=(5.5, 3.6)); plt.plot(front[:, 0], front[:, 1], "o-")
plt.xlabel("mass (kg)"); plt.ylabel("tip deflection (mm)"); plt.title("Pareto front of the beam"); plt.show()

### Your turn, exercise 3

Count the non-dominated points among the five (mass, deflection) pairs below. A pair dominates another if it is no worse in both values and better in at least one. Store the count in `n_front`.

In [ ]:
pairs = [(120, 7.5), (135, 6.0), (130, 7.0), (160, 4.8), (150, 6.2)]

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_front_reference():
    dom = lambda a, b: a[0] <= b[0] and a[1] <= b[1] and a != b
    return sum(not any(dom(q, p) for q in pairs) for p in pairs)

In [ ]:
n_front = None   # your computation
check("Exercise 3", n_front, _n_front_reference())

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-05/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-05/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-05/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 6](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-06/README.md) starts with its overview.

## References

[1] Gere, J. M., & Goodno, B. J. (2013). *Mechanics of Materials* (8th ed.). Cengage Learning.

[2] Virtanen, P., Gommers, R., Oliphant, T. E., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods*, *17*(3), 261-272. <https://doi.org/10.1038/s41592-019-0686-2>

[3] Holland, J. H. (1992). *Adaptation in Natural and Artificial Systems* (2nd ed.). MIT Press. First edition 1975, University of Michigan Press.

[4] Goldberg, D. E. (1989). *Genetic Algorithms in Search, Optimization, and Machine Learning*. Addison-Wesley.

[5] Eiben, A. E., & Smith, J. E. (2015). *Introduction to Evolutionary Computing* (2nd ed.). Springer. <https://doi.org/10.1007/978-3-662-44874-8>

[6] Storn, R., & Price, K. (1997). Differential evolution: A simple and efficient heuristic for global optimization over continuous spaces. *Journal of Global Optimization*, *11*(4), 341-359. <https://doi.org/10.1023/A:1008202821328>

[7] Deb, K. (2000). An efficient constraint handling method for genetic algorithms. *Computer Methods in Applied Mechanics and Engineering*, *186*(2-4), 311-338. <https://doi.org/10.1016/S0045-7825(99)00389-8>

[8] Deb, K., Pratap, A., Agarwal, S., & Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm: NSGA-II. *IEEE Transactions on Evolutionary Computation*, *6*(2), 182-197. <https://doi.org/10.1109/4235.996017>